# VECTOR DB

In [1]:
from qdrant_client import QdrantClient
import os
import qdrant_client
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex, StorageContext, Settings, SimpleDirectoryReader, SummaryIndex
from llama_index.core.schema import TextNode
from llama_index.embeddings.openai import OpenAIEmbedding
from dotenv import load_dotenv
from qdrant_client.http.models import (
    VectorParams,
    SparseVectorParams,
    Distance
)
import json
import logging
import time
from pathlib import Path
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    granite_picture_description
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import ImageRefMode, PictureItem 
from docling.chunking import HybridChunker, HierarchicalChunker
from docling.datamodel.document import DoclingDocument
from llama_index.llms.openai_like import OpenAILike
#qua fatto porcata per inserire utils
import importlib
import utils as utils
utils = importlib.reload(utils)
from utils import iniezionetagimmagini, riassuntodocumento, metadata_extraction
import sqlite3

load_dotenv()

True

## SETTINGS GENERALI

In [2]:
collectionname = "WAMASRAGAGENT"

url_embedder = os.getenv("VLLM_API_BASE_URL")

url_qdrant = os.getenv("QDRANT_URL")

client = qdrant_client.QdrantClient(url=url_qdrant)

embed_model = OpenAIEmbedding(
    api_base=url_embedder,
    model_name="BAAI/bge-m3",
    api_key="null",
)

Settings.embed_model = embed_model

# Vector store
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collectionname,
    enable_hybrid=True,
    dense_vector_name="bge_m3",
    sparse_vector_name="bm25",
    fastembed_sparse_model="Qdrant/bm25"
)

storage_context = StorageContext.from_defaults(vector_store=vector_store)


2026-01-29 08:10:55,686 - INFO - HTTP Request: GET http://10.1.1.193:6333 "HTTP/1.1 200 OK"
2026-01-29 08:10:56,383 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASRAGAGENT/exists "HTTP/1.1 200 OK"
2026-01-29 08:10:56,424 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASRAGAGENT "HTTP/1.1 200 OK"
2026-01-29 08:10:56,429 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASRAGAGENT/exists "HTTP/1.1 200 OK"
2026-01-29 08:10:56,431 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASRAGAGENT "HTTP/1.1 200 OK"
2026-01-29 08:10:56,491 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASRAGAGENT/exists "HTTP/1.1 200 OK"
2026-01-29 08:10:56,495 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASRAGAGENT "HTTP/1.1 200 OK"


## Fare solo una volta!!! (o per testing)

In [3]:
if client.collection_exists(collectionname):
    client.delete_collection(collectionname)

client.create_collection(
    collection_name=collectionname,
    vectors_config={
        "bge_m3": VectorParams(
            size=1024,   
            distance=Distance.COSINE
        )
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams()
    }
)

2026-01-29 08:10:56,555 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASRAGAGENT/exists "HTTP/1.1 200 OK"
2026-01-29 08:10:56,606 - INFO - HTTP Request: DELETE http://10.1.1.193:6333/collections/WAMASRAGAGENT "HTTP/1.1 200 OK"
2026-01-29 08:10:56,873 - INFO - HTTP Request: PUT http://10.1.1.193:6333/collections/WAMASRAGAGENT "HTTP/1.1 200 OK"


True

In [4]:
if os.path.exists(f"../{collectionname}.db"):
    os.remove(f"../{collectionname}.db")

conn = sqlite3.connect(f"../{collectionname}.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS document_chunks (
        filename TEXT,
        chunk_index INTEGER,
        text_content TEXT,
        PRIMARY KEY (filename, chunk_index)
    )
""")
conn.commit()

## FUNZIONE PER EMBEDDARE CON TAGIMAGE + OVERLAP

In [5]:
from llama_index.core import Document, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import TextNode

def qdrantembedding(DOC_SOURCE, DOC_SOURCE_MD):
    
    doc_dl = DoclingDocument.load_from_json(DOC_SOURCE)

    DOC_SOURCE = DOC_SOURCE.split("/")[-1].replace(".json", ".pdf")
    gruppi = DOC_SOURCE.split("_")[0]
    gruppi = gruppi.split("-") if "-" in gruppi else [gruppi]
    
    
    doc_dl = iniezionetagimmagini(doc_dl)

    full_text = doc_dl.export_to_markdown()

    full_text = full_text.replace("<!-- image -->", "")
 
    
    base_metadata = {"origin_filename": DOC_SOURCE, "groups": gruppi}
    
    
    llama_doc = Document(text=full_text, metadata=base_metadata)

    splitter = SentenceSplitter(
        chunk_size=256, 
        chunk_overlap=20
    )

    nodes = splitter.get_nodes_from_documents([llama_doc])
    
    sql_data = []

    for i, node in enumerate(nodes):
        
        enriched_text = node.get_content() 

        clean_meta = node.metadata.copy()
        clean_meta["chunk_index"] = i
        
        node.metadata = clean_meta

        sql_data.append((clean_meta.get("origin_filename", ""), i, enriched_text))


    cursor.executemany(
        "INSERT OR REPLACE INTO document_chunks (filename, chunk_index, text_content) VALUES (?, ?, ?)", 
        sql_data
    )
    conn.commit()

    index = VectorStoreIndex(
        nodes,
        storage_context=storage_context,
        show_progress=True
    )

    print(f"Documento {DOC_SOURCE} embeddato con chunk da 1000 token e overlap 200.")

Cuore dello script

In [6]:
base_folder = "../preprocessing/scratch"


json_file = ""
md_file = ""


for root, dirs, files in os.walk(base_folder):
    for file in files:
        if file.endswith(".json"):
            json_file = os.path.join(root, file)
        elif file.endswith(".md"):
            md_file = os.path.join(root, file)
    if json_file and md_file:
        #print(json_file)
        qdrantembedding(DOC_SOURCE=json_file, DOC_SOURCE_MD=md_file)



2026-01-29 08:11:01,461 - INFO - Migrating deprecated `annotations` to `meta`; this will be removed in the future. Note that only the first available instance of each annotation type will be migrated.
2026-01-29 08:11:01,461 - INFO - Migrating deprecated `annotations` to `meta`; this will be removed in the future. Note that only the first available instance of each annotation type will be migrated.
2026-01-29 08:11:01,462 - INFO - Migrating deprecated `annotations` to `meta`; this will be removed in the future. Note that only the first available instance of each annotation type will be migrated.
2026-01-29 08:11:01,462 - INFO - Migrating deprecated `annotations` to `meta`; this will be removed in the future. Note that only the first available instance of each annotation type will be migrated.
2026-01-29 08:11:01,462 - INFO - Migrating deprecated `annotations` to `meta`; this will be removed in the future. Note that only the first available instance of each annotation type will be migra

Generating embeddings:   0%|          | 0/12 [00:00<?, ?it/s]

2026-01-29 08:11:01,782 - INFO - HTTP Request: POST http://10.1.2.98/v1/embeddings "HTTP/1.1 200 OK"
2026-01-29 08:11:01,794 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASRAGAGENT "HTTP/1.1 200 OK"
2026-01-29 08:11:01,866 - INFO - HTTP Request: PUT http://10.1.1.193:6333/collections/WAMASRAGAGENT/points?wait=true "HTTP/1.1 200 OK"
2026-01-29 08:11:01,875 - INFO - Migrating deprecated `annotations` to `meta`; this will be removed in the future. Note that only the first available instance of each annotation type will be migrated.
2026-01-29 08:11:01,875 - INFO - Migrating deprecated `annotations` to `meta`; this will be removed in the future. Note that only the first available instance of each annotation type will be migrated.
2026-01-29 08:11:01,876 - INFO - Migrating deprecated `annotations` to `meta`; this will be removed in the future. Note that only the first available instance of each annotation type will be migrated.
2026-01-29 08:11:01,876 - INFO - Migrating 

Documento UTL_Refresh_Reset_OT.pdf embeddato con chunk da 1000 token e overlap 200.


Generating embeddings:   0%|          | 0/10 [00:00<?, ?it/s]

...

2026-01-29 08:11:03,126 - INFO - HTTP Request: POST http://10.1.2.98/v1/embeddings "HTTP/1.1 200 OK"
2026-01-29 08:11:03,131 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASRAGAGENT "HTTP/1.1 200 OK"
2026-01-29 08:11:03,168 - INFO - HTTP Request: PUT http://10.1.1.193:6333/collections/WAMASRAGAGENT/points?wait=true "HTTP/1.1 200 OK"
2026-01-29 08:11:03,183 - INFO - Migrating deprecated `annotations` to `meta`; this will be removed in the future. Note that only the first available instance of each annotation type will be migrated.
2026-01-29 08:11:03,184 - INFO - Migrating deprecated `annotations` to `meta`; this will be removed in the future. Note that only the first available instance of each annotation type will be migrated.
2026-01-29 08:11:03,184 - INFO - Migrating deprecated `annotations` to `meta`; this will be removed in the future. Note that only the first available instance of each annotation type will be migrated.
2026-01-29 08:11:03,184 - INFO - Migrating 

Documento UTL_Rimozione_manuale_PLC_SOC.pdf embeddato con chunk da 1000 token e overlap 200.


Generating embeddings:   0%|          | 0/19 [00:00<?, ?it/s]

2026-01-29 08:11:03,244 - INFO - HTTP Request: POST http://10.1.2.98/v1/embeddings "HTTP/1.1 200 OK"
2026-01-29 08:11:03,250 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASRAGAGENT "HTTP/1.1 200 OK"
2026-01-29 08:11:03,325 - INFO - HTTP Request: PUT http://10.1.1.193:6333/collections/WAMASRAGAGENT/points?wait=true "HTTP/1.1 200 OK"


Documento LOG_Creazione_carico.pdf embeddato con chunk da 1000 token e overlap 200.
